#### **1. INITIALIZATION**

In [21]:
# Install required libraries just in case Colab doesn't have them updated
!pip install -q transformers torch pillow

In [22]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [23]:
import torch
import torch.nn as nn
from transformers import CLIPVisionModel, CLIPImageProcessor
from PIL import Image

#### **2. REPLACE THE TEXT LAYER WITH CLASIFICATION LAYER**

In [24]:
import torch.nn as nn
from transformers import CLIPVisionModel

class ClassificationCLIP(nn.Module):
    def __init__(self, model_path):
        super(ClassificationCLIP, self).__init__()

        # Step 1: Load only the Vision part (Completely discard the Text Encoder)
        print("Loading CLIP Vision Encoder...")
        self.vision_encoder = CLIPVisionModel.from_pretrained(model_path)

        # Get the output vector size of the CLIP vision encoder (1024 for clip-vit-large-patch14)
        hidden_size = self.vision_encoder.config.hidden_size

        # Step 2: Initialize the Classification layer (Linear Layer)
        # Input is 1024 features, Output is a single number (Logit) for BCE Loss
        print("Attaching Classification Head...")
        self.classifier = nn.Linear(hidden_size, 1)

    def forward(self, pixel_values):
        # Pass the image through the Vision Encoder
        outputs = self.vision_encoder(pixel_values=pixel_values)

        # Extract the global feature representation (1D vector representing the entire image)
        pooled_output = outputs.pooler_output

        # Pass the extracted features through the Classifier to get the final logit
        logits = self.classifier(pooled_output)

        return logits

#### **3. INITIALIZE AND SANITY CHECK**

In [14]:
# Model path
save_path = "/content/drive/MyDrive/Model/clip_model"

# Innitialize new model
model_baseline = ClassificationCLIP(save_path)
processor = CLIPImageProcessor.from_pretrained(save_path)

Loading CLIP Vision Encoder...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

CLIPVisionModel LOAD REPORT from: /content/drive/MyDrive/Model/clip_model
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
logit_scale                                                  | UNEXPECTED |  | 
text_model.embeddings.position_embedding.weight              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bia

Attaching Classification Head...


In [15]:
# Temporarily set the model to evaluation mode
model_baseline.eval()

print("\nRunning a test on a single image through the new architecture...")
try:
    # Take a random image out to process
    image = Image.open("/content/drive/MyDrive/TrainingData/vision/D01_Samsung_GalaxyS3Mini/D01_I_flat_0001.jpg")

    # Process the image (Text input is no longer needed!)
    inputs = processor(images=image, return_tensors="pt")

    with torch.no_grad():
        # The output is now a single raw score (Logit)
        logit = model_baseline(inputs.pixel_values)

        # Compress the raw score into the 0 -> 1 range (0% to 100%) using the Sigmoid function
        prob = torch.sigmoid(logit)

    print(f"Run successful! Untrained result (Fake Probability): {prob.item() * 100:.2f}%")
    print("Note: The current result is random because the Classification layer has not been trained yet!")

except FileNotFoundError:
    print("Error: File not found!")


Running a test on a single image through the new architecture...
Run successful! Untrained result (Fake Probability): 44.30%
Note: The current result is random because the Classification layer has not been trained yet!


#### **4. SAVE CLASSIFICATION WEIGHTS TO FOLDER**

In [18]:
# 1. Create a NEW directory to store the training results
trained_folder = "/content/drive/MyDrive/Model/clip_classification_weights"
os.makedirs(trained_folder, exist_ok=True)

# 2. ONLY extract the weights of the classification layer to save
# This file will have a .pth or .pt extension and a very small file size (only a few MBs)
classifier_weights_path = os.path.join(trained_folder, "classifier_weights.pth")
torch.save(model_baseline.classifier.state_dict(), classifier_weights_path)

print(f"Successfully saved the new classification weights to: {classifier_weights_path}")

Successfully saved the new classification weights to: /content/drive/MyDrive/Model/clip_classification_weights/classifier_weights.pth


#### **5. LOAD WEIGHTS READY TO TRAIN/TEST**

In [30]:
# 1. Paths to the base model and trained weights directories
base_clip_folder = "/content/drive/MyDrive/Model/clip_model"
trained_folder = "/content/drive/MyDrive/Model/clip_classification_weights"

# 2. Initialize the base architecture (Loads the 1.7GB Base Model)
print("Constructing the base model architecture from CLIP...")
my_model = ClassificationCLIP(base_clip_folder)

# 3. Attach the trained classification head (Loads and overwrites with the trained weights)
print("Loading and applying the trained classification weights...")
trained_weights = torch.load(os.path.join(trained_folder, "classifier_weights.pth"))
my_model.classifier.load_state_dict(trained_weights)

print("\n")
print("------- Assembly complete! The model is ready for evaluation -------")

Constructing the base model architecture from CLIP...
Loading CLIP Vision Encoder...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

CLIPVisionModel LOAD REPORT from: /content/drive/MyDrive/Model/clip_model
Key                                                          | Status     |  | 
-------------------------------------------------------------+------------+--+-
text_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
logit_scale                                                  | UNEXPECTED |  | 
text_model.embeddings.position_embedding.weight              | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.out_proj.weight | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.k_proj.bias     | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.self_attn.q_proj.weight   | UNEXPECTED |  | 
text_model.encoder.layers.{0...11}.mlp.fc1.bia

Attaching Classification Head...
Loading and applying the trained classification weights...


------- Assembly complete! The model is ready for evaluation -------


#### **6. TRAINING MODEL**